In [ ]:
"""
Add description of project here
"""


### Installing needed packages

In [ ]:
#pip install langchain-cohere

In [ ]:
# https://github.com/googlecolab/colabtools/issues/5455
# For langchain-cohere==0.4.4 downgrade cohere to 5.15.0, it will solve the problem.

#%pip uninstall cohere
#%pip install cohere==5.15.0

### Import packages and env vars

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()  # Loads from .env
cohere_api_key=os.getenv("COHERE_TOKEN")

In [2]:
from langchain_cohere import ChatCohere
#from langchain_core.messages import AIMessage, HumanMessage

In [3]:
# Memory start
from langchain.memory import ConversationSummaryMemory
from langchain import PromptTemplate
from langchain import LLMChain


### Define arguments for chatbot

In [13]:
# Define the Cohere LLM
llm = ChatCohere(
    cohere_api_key=cohere_api_key, model="command-a-03-2025"
)

In [14]:
# Create an updated prompt template to include a chat history
# template = """<s><|user|>Current conversation:{chat_history}

# {input_prompt}<|end|>
# <|assistant|>"""

template = """Current conversation:{chat_history}

{input_prompt}"""

prompt = PromptTemplate(
    template=template,
    input_variables=["input_prompt", "chat_history"]
)


In [15]:
#structure that cohere uses
#messages=[{"role": "user", "content": "hello world!"}]

In [ ]:

# Create a summary prompt template
# I dont know if the tokens <s><|user|> are useful for this model. They are not!
# summary_prompt_template = """<s><|user|>Summarize the conversations and update with the new lines.

# Current summary:
# {summary}

# new lines of conversation:
# {new_lines}

# New summary:<|end|>
# <|assistant|>"""

# summary_prompt_template = """Summarize the conversations and update with the new lines.

# Current summary:
# {summary}

# new lines of conversation:
# {new_lines}

# New summary:"""


summary_prompt_template = """Create a new summary with the current summary and the new lines of conversation. The new summary will be used as memory for a chatbot assistant.

Current summary:
{summary}

new lines of conversation:
{new_lines}"""


summary_prompt = PromptTemplate(
    input_variables=["new_lines", "summary"],
    template=summary_prompt_template
)



In [35]:
# Define the type of memory we will use
memory = ConversationSummaryMemory(
    llm=llm,
    memory_key="chat_history",
    prompt=summary_prompt
)



### Define chain

In [36]:
# Chain the LLM, prompt, and memory together
llm_chain = LLMChain(
    prompt=prompt,
    llm=llm,
    memory=memory
)

### Test it

In [37]:
# Generate a conversation and ask for the name
llm_chain.invoke({"input_prompt": "Hi! My name is Carlos. What is 1 + 4?"})


{'input_prompt': 'Hi! My name is Carlos. What is 1 + 4?',
 'chat_history': '',
 'text': "Hi Carlos! It's nice to meet you. The answer to 1 + 4 is **5**. How can I assist you further?"}

In [38]:
llm_chain.invoke({"input_prompt": "What is my name?"})

{'input_prompt': 'What is my name?',
 'chat_history': "**Updated Summary:**\n\nCarlos introduced himself and asked for the sum of 1 + 4. The AI greeted Carlos, provided the correct answer of **5**, and offered further assistance.  \n\n**Memory for Chatbot Assistant:**  \n- User's name: Carlos  \n- Last interaction: Carlos asked for the sum of 1 + 4, and the AI responded with **5** and offered additional help.",
 'text': 'Your name is Carlos. How can I assist you further today?'}

In [39]:
# Check whether it has summarized everything thus far
llm_chain.invoke({"input_prompt": "What was the first question I asked?"})

{'input_prompt': 'What was the first question I asked?',
 'chat_history': '**Updated Summary:**\n\nCarlos introduced himself and asked for the sum of 1 + 4. The AI greeted Carlos, provided the correct answer of **5**, and offered further assistance. Carlos then asked the AI to confirm his name, and the AI correctly recalled his name as Carlos, reiterating its availability for further help.\n\n**Memory for Chatbot Assistant:**  \n- User\'s name: Carlos  \n- Last interaction: Carlos asked the AI to confirm his name, and the AI correctly responded with "Your name is Carlos," offering continued assistance.  \n- Previous interaction: Carlos asked for the sum of 1 + 4, and the AI responded with **5** and offered additional help.',
 'text': 'The first question you asked was for the sum of 1 + 4.'}

### Access memory

In [40]:
# Check what the summary is thus far
memory.load_memory_variables({})

{'chat_history': '**Updated Summary:**\n\nCarlos introduced himself and asked for the sum of 1 + 4. The AI greeted Carlos, provided the correct answer of **5**, and offered further assistance. Carlos then asked the AI to confirm his name, and the AI correctly recalled his name as Carlos, reiterating its availability for further help. Later, Carlos asked what his first question was, and the AI accurately responded that it was for the sum of 1 + 4.\n\n**Memory for Chatbot Assistant:**  \n- User\'s name: Carlos  \n- Last interaction: Carlos asked what his first question was, and the AI correctly responded that it was for the sum of 1 + 4.  \n- Previous interactions:  \n  1. Carlos asked for the sum of 1 + 4, and the AI responded with **5** and offered additional help.  \n  2. Carlos asked the AI to confirm his name, and the AI correctly responded with "Your name is Carlos," offering continued assistance.'}